# How to use Spectral Filtering on any dataset

Given a **dataset** and a large **concept dictionary**, spectral filtering identifies
the small subset of concepts that best describe the visual content of the dataset.

### How it works
1. Encode every image and every concept with CLIP.
2. Build a matrix of image–concept similarities (softmax-normalised).
3. Eigendecompose the covariance of that matrix.
4. Score each concept by how much it contributes to the top principal components.
5. Keep only the concepts whose cumulative importance exceeds a threshold `beta_c`.

In [ ]:
import os, sys

# Set the project root directory and ensure it's in the Python path for imports
PROJECT_ROOT = "path_to_project_root/spectralgcd"
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
import open_clip
from clip import tokenize

# spectral_filter implements the Spectral Filtering pipeline described in Sec. 3.2
from spectral_filtering_function import spectral_filter
from data.fgvc_aircraft import FGVCAircraft

## Configuration

In [ ]:
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Paths
AIRCRAFT_ROOT      = "path_to_dataset/fgvc_aircraft"
# Concept dictionary D (TextGCD Tags, ~22K concepts).
PATH_TO_DICTIONARY = "dictionaries/textgcd_tags_dictionary.csv"
PATH_TO_OUTPUT     = "./filtered_concepts_demo/aircraft_concepts.csv"

# Teacher model: ViT-H/14 trained on LAION-2B (default teacher in Table 1).
CLIP_MODEL = "hf-hub:laion/CLIP-ViT-H-14-laion2B-s32B-b79K"

# Spectral Filtering thresholds:
#   beta_e — fraction of cumulative eigenvalue variance to retain when
#             truncating the eigen-spectrum of the cross-modal covariance.
#   beta_c — fraction of cumulative concept-importance scores to retain
#             when selecting the final concept subset.
# Default values beta_e=0.95, beta_c=0.99 used for all main results.
BETA_E      = 0.95
BETA_C      = 0.99
# Temperature for the softmax over image-concept similarities.
# Low temperature sharpens each image's distribution over concepts.
TEMPERATURE = 0.01

# Data loading
BATCH_SIZE  = 128
NUM_WORKERS = 4

## 1 – Load CLIP

In [ ]:
print(f"Loading {CLIP_MODEL} ...")
model_clip, _, preprocess = open_clip.create_model_and_transforms(CLIP_MODEL)
model_clip = model_clip.to(DEVICE).eval()

# The teacher is kept fully frozen throughout Spectral Filtering — its
# cross-modal similarities define the covariance structure we analyse.
for p in model_clip.parameters():
    p.requires_grad = False

# Probe the image embedding dimensionality (1024 for ViT-H/14).
with torch.no_grad():
    feat_dim = model_clip.encode_image(torch.zeros(1, 3, 224, 224, device=DEVICE)).shape[-1]

print(f"Ready  --  feature dim: {feat_dim}")

## 2 – Load Concept Dictionary and Extract Text Features

In [ ]:
# Load the concept dictionary D.
concept_names = pd.read_csv(PATH_TO_DICTIONARY)["ClassName"].astype(str).tolist()
print(f"Concepts in dictionary: {len(concept_names)}")
print(f"Sample: {concept_names[:5]}")

In [ ]:
text_features = []
model_clip.eval()

# Encode every concept with the standard "a photo of a {concept} ." prompt template.
# These L2-normalised embeddings form the columns of the similarity matrix S
# (image x concept) used to build the cross-modal covariance.
with torch.no_grad():
    for i in tqdm(range(0, len(concept_names), 512), desc="Text encoding"):
        batch   = concept_names[i : i + 512]
        prompts = [f"a photo of a {n} ." for n in batch]
        tok     = tokenize(prompts).to(DEVICE)
        feats   = model_clip.encode_text(tok).float()
        text_features.append(feats.cpu())

text_features = torch.cat(text_features, dim=0)   # shape: (|D|, feat_dim)
print(f"Text features: {text_features.shape}")

## 3 – Load Aircraft Dataset and Extract Image Features

We load all training images (`split='trainval'`).

In [ ]:
# Load the dataset whose concept distribution we want to analyse.
# Spectral Filtering is data-driven: it uses the actual image distribution to
# identify which concepts from D are truly relevant for this dataset.
dataset = FGVCAircraft(root=AIRCRAFT_ROOT, split="trainval", transform=preprocess)
print(f"Dataset size: {len(dataset)} images")

loader = torch.utils.data.DataLoader(
    dataset,
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    shuffle     = False,
    drop_last   = False,
    pin_memory  = True,
)

In [ ]:
image_features = []
model_clip.eval()

# Extract L2-normalised image embeddings from the frozen teacher.
with torch.no_grad():
    for images, *_ in tqdm(loader, desc="Image encoding"):
        feats = model_clip.encode_image(images.to(DEVICE)).float()
        image_features.append(feats.cpu())

image_features = torch.cat(image_features, dim=0)   # shape: (N, feat_dim)
print(f"Image features: {image_features.shape}")

## 4 – Run Spectral Filtering

In [ ]:
# Run the full Spectral Filtering pipeline:
#   1. Compute softmax-normalised image-concept similarity matrix S (N x |D|).
#   2. Build the cross-modal covariance Sigma = S^T S / N over concept axis.
#   3. Eigendecompose Sigma; retain the top eigenvectors that explain
#      beta_e=95% of the total variance.
#   4. Project each concept onto the retained eigenbasis and compute an
#      importance score.
#   5. Sort concepts by importance and keep those whose cumulative score
#      reaches beta_c=99% of the total — the filtered dictionary D*.
#
# normalize_features=True applies L2 normalisation before similarity computation.
# apply_softmax=True activates the temperature-scaled softmax.
ordered_concepts, n_retained = spectral_filter(
    image_features     = image_features.numpy(),
    text_features      = text_features.numpy(),
    concept_names      = concept_names,
    normalize_features = True,
    temperature        = TEMPERATURE,
    beta_e             = BETA_E,
    beta_c             = BETA_C,
    apply_softmax      = True,
    device             = str(DEVICE),
    batch_size         = 200,
)

# ordered_concepts: full dictionary sorted by decreasing importance score.
# n_retained: number of concepts in the filtered subset D* (index cutoff).
print(f"Dictionary size  : {len(concept_names)}")
print(f"Concepts retained: {n_retained}  ({n_retained / len(concept_names):.1%})")

## 5 – Inspect Results

In [ ]:
# Display the top-30 concepts by importance score.
print(f"{'Rank':<5} Concept")
print("-" * 50)
for rank, name in enumerate(ordered_concepts[:30], 1):
    print(f"{rank:<5} {name}")

In [ ]:
# Save filtered concepts.
os.makedirs(os.path.dirname(PATH_TO_OUTPUT), exist_ok=True)
pd.DataFrame(ordered_concepts[:n_retained], columns=["ClassName"]).to_csv(PATH_TO_OUTPUT, index=False)
print(f"Saved {n_retained} concepts -> {PATH_TO_OUTPUT}")